In [1]:
import os
import re
import random
import glob
import math
import pandas as pd
import numpy as np

In [2]:
BASE_CONFIG_PATH = "../resources/config_test_scale_upd_bc_depo.txt"
OUTPUT_DIR = "configs_generated500_abs_2026_03_09/"
FOREST_OUTPUT_DIR = "../forest-generator/output500_abs_2026_03_09/"  # Path where forest/roads txts are

# Ranges
ALPHA = (0.0, 2*np.pi) 
V_RANGE = (0.5, 15.0)
# T_SURF_RANGE = (263.0, 300.0)
VELOCITY_CONST = 3.0
dPdX_CONST = -0.00025
random.seed(61) # 42, 22

In [ ]:
# def parse_road_configs(filepath):
#     """
#     Parses the road output file.
#     Expected format:
#     # Config 0
#     xmin, xmax, ymin, ymax
#     ...
#     # Config 1
#     ...
    
#     Returns: A dictionary {config_id: {'xmin': float, ...}}
#     If multiple lines exist for one config, it calculates the bounding box.
#     """
#     configs = {}
#     current_id = -1
#     current_rects = []

#     if not os.path.exists(filepath):
#         print(f"Warning: Road file {filepath} not found.")
#         return {}

#     with open(filepath, 'r') as f:
#         for line in f:
#             line = line.strip()
#             if not line: continue
            
#             # Detect Config Header
#             m = re.match(r'# Config (\d+)', line)
#             if m:
#                 # Save previous if exists
#                 if current_id != -1 and current_rects:
#                     configs[current_id] = get_bounding_box(current_rects)
                
#                 current_id = int(m.group(1))
#                 current_rects = []
#                 continue
            
#             # Detect Coordinates
#             # Expecting: xmin, xmax, ymin, ymax
#             # Allow comma or space separation
#             parts = re.split(r'[,\s]+', line)
#             # Filter empty strings
#             parts = [p for p in parts if p]
            
#             if len(parts) >= 4 and not line.startswith('#'):
#                 try:
#                     rect = {
#                         'xmin': float(parts[0]), 'xmax': float(parts[1]),
#                         'ymin': float(parts[2]), 'ymax': float(parts[3])
#                     }
#                     current_rects.append(rect)
#                 except ValueError:
#                     pass

#         # Save last one
#         if current_id != -1 and current_rects:
#             configs[current_id] = get_bounding_box(current_rects)
            
#     return configs

In [3]:
def parse_road_configs(filepath):
    """
    Parses the road output file.
    Returns: A dictionary {config_id: [rect1, rect2, ...]}
    Where rect is {'xmin': float, ...}
    """
    configs = {}
    current_id = -1
    current_rects = []

    if not os.path.exists(filepath):
        print(f"Warning: Road file {filepath} not found.")
        return {}

    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            
            # Detect Config Header
            m = re.match(r'# Config (\d+)', line)
            if m:
                # Save previous if exists
                if current_id != -1:
                    configs[current_id] = current_rects
                
                current_id = int(m.group(1))
                current_rects = []
                continue
            
            parts = re.split(r'[,\s]+', line)
            parts = [p for p in parts if p]
            
            if len(parts) >= 4 and not line.startswith('#'):
                try:
                    rect = {
                        'xmin': float(parts[0]), 'xmax': float(parts[1]),
                        'ymin': float(parts[2]), 'ymax': float(parts[3])
                    }
                    current_rects.append(rect)
                except ValueError:
                    pass

        # Save last one
        if current_id != -1:
            configs[current_id] = current_rects
            
    return configs

In [ ]:
# def get_bounding_box(rects):
#     """Returns the union bounding box of a list of rectangles."""
#     if not rects:
#         return {'xmin': 0.0, 'xmax': 1.0, 'ymin': 0.0, 'ymax': 1.0}
    
#     return {
#         'xmin': min(r['xmin'] for r in rects),
#         'xmax': max(r['xmax'] for r in rects),
#         'ymin': min(r['ymin'] for r in rects),
#         'ymax': max(r['ymax'] for r in rects)
#     }

# def generate_tracer_block(tracer_id, road_rect):
#     """Generates the string block for a single passive tracer."""
#     # Note: Indentation is kept to match config style
#     return f"""
# 	tracer_{tracer_id} {{
# 		diffusivity = phys.xi;
# 		type = "box";

# 		density = 1.0 * 1000.0;    # [kg / m^3]
# 		diameter = 2.5 * 0.000001;  # [m]
		
# 		zmin = passive_tracers.emission.zmin; 
# 		zmax = passive_tracers.emission.zmax;
			
# 		value = domain.length * 20.0;
# 		value = value * passive_tracers.emission.coeff * 0.17;
		
# 		normalize_value = passive_tracers.emission.coeff * passive_tracers.emission.factor_footway * 0.17;
		
# 		normalize_value = normalize_value * (domain.length / grid.cx) * (domain.width / grid.cy);
		
# 		surface {{ 
# 			flux = -0.01;

# 			# --- setting localized source based on road config
# 			xmin = {road_rect['xmin']:.4f}; xmax = {road_rect['xmax']:.4f};
# 			ymin = {road_rect['ymin']:.4f}; ymax = {road_rect['ymax']:.4f};

# 			# --- active in [begin, end]
# 			begin = 1.0 * 3600.0;
# 		}}

# 		les {{
# 			dynamic {{ 
# 				# avg_mode = "lagrange";
# 			}}
# 		}}
# 	}}
# """

In [4]:
def generate_tracer_block(tracer_id, road_rects):
    """
    Generates a tracer block with multiple point_emission sources based on road segments.
    """
    
    # 1. Header
    block = f"""
    tracer_{tracer_id} {{ 
        diffusivity = phys.xi;
        density = 1.0 * 1000.0;    # [kg / m^3]
		diameter = 3.0 * 0.000001;  # [m]
		
		# --- min limit [optional, default = none]
		min_value = 0.0;
        # --- setting 'life-time' using decay frequency [1/s] [optional]
        # f_decay = 1.0 / 600.0;		

        # --- remove mean from concentration profile [optional, default = false]
        # is_remove_mean = false;

        # --- sedimentation & deposition [optional, default = false]
        # 	--- require density & diameter setup
        is_sedimentation = true;
        is_dry_deposition = true;
        # is_wet_deposition = false;

        #	--- specialize dry deposition, default = general key value
        # is_dry_surface_deposition = true;
        # is_dry_roof_deposition = true;
        # is_dry_walls_deposition = true;		

        is_canopy_settling_deposition = true;
        is_canopy_brownian_deposition = true;
        is_canopy_impaction_deposition = true;
        is_canopy_interception_deposition = true;
        # is_canopy_generic_deposition = false;	


        surface {{ 
            flux = -0.0;
        }}
"""

    # 2. Point Emissions (Roads)
    num_roads = len(road_rects)
    if num_roads == 0:
        block += "\t\tnum_point_emission = 0;\n"
    else:
        block += f"\t\tnum_point_emission = {num_roads};\n\n"
        
        for idx, rect in enumerate(road_rects, 1):
            block += f"""\t\tpoint_emission_{idx}
		{{
			type = "box";

			xmin = {rect['xmin']:.4f}; xmax = {rect['xmax']:.4f};
			ymin = {rect['ymin']:.4f}; ymax = {rect['ymax']:.4f};

			zmin = passive_tracers.emission.zmin; 
			zmax = passive_tracers.emission.zmax;

			value = (xmax - xmin) * (ymax - ymin) *  (ymax - ymin) * (zmax -zmin);

			normalize_value = passive_tracers.emission.coeff;			
		}}
"""

    # 3. Footer (LES settings)
    block += """
		les {
            # --- specialize subgrid model, optional, default = scalar
            # diff_model = "ref";
            # is_ssm_mixed = false;

            # C_ssm = 1.0;
            
            # Schmidt_sgs = 0.7;

            dynamic { 
                # Schmidt_sgs_min = 0.3;

                # alpha = 1.73;		# test-to-base filter width ratio
                
                # avg_mode = "lagrange";	# "none", "plane", "filter", "lagrange"
                
                # nskip = 3;
                # use_transport = false;				
                
                # C_T_lagrange = 3.0;
            }
        }
        # --- boundary conditions [optional]
        boundary_conditions
        {
            # --- default: -xy periodic & neumann (= 0) at top and bottom
            west { 
                type = "inout"; 
                rhs = 0.0;
            }
            east { 
                type = "inout"; 
                rhs = 0.0; 				
            }
            south { 
                type = "inout"; 
                rhs = 0.0; 				
            }
            north { 
                type = "inout"; 
                rhs = 0.0; 			
            }
        }
    }
"""
    return block

In [5]:
def process_config_file(exp_id, dPdx_val, dPdy_val,u_val, v_val, road_data, input_path, output_path):
    with open(input_path, 'r') as f_in, open(output_path, 'w') as f_out:
        in_geo_wind = False
        in_canopy_list = False
        in_passive_tracers = False
        skip_lines_until_brace = False
        passive_tracers_brace_count = 0
        in_initial_condition = False
        in_U_in_initial_condition = False
        in_V_in_initial_condition = False
        iterator = iter(f_in)
        for line in iterator:
            stripped = line.strip()
            
            # 1. external_pressure_grad
            if 'external_pressure_grad' in stripped:
                in_geo_wind = True
                f_out.write(line)
                continue
            if in_geo_wind:
                if '}' in stripped:
                    in_geo_wind = False
                elif 'dPdx = ' in line:
                    f_out.write(f"\t dPdx =  {dPdx_val + 1e-6:.6f}; \t\t# [Pa/m]\n")
                    continue
                elif 'dPdy = ' in line:
                    f_out.write(f"\t dPdy =  {dPdy_val + 1e-6:.6f}; \t\t# [Pa/m]\n")
                    continue
            
            # 2. INITIAL CONDITIONS (THETA)
            if 'initial_conditions' in stripped:
                in_initial_condition = True
                f_out.write(line)
                continue
            if in_initial_condition and 'U {' in stripped:
                in_U_in_initial_condition = True
                f_out.write(line)
                continue
            if in_initial_condition and in_U_in_initial_condition and 'bulk' in stripped:
                f_out.write(f"\t\tbulk = {u_val:.2f};\n")
                continue
            if in_initial_condition and in_U_in_initial_condition and '}' in stripped:
                in_U_in_initial_condition = False
                f_out.write(line)
                continue
            if in_initial_condition and 'V {' in stripped:
                in_V_in_initial_condition = True
                f_out.write(line)
                continue
            if in_initial_condition and in_V_in_initial_condition and 'bulk' in stripped:
                f_out.write(f"\t\tbulk = {v_val:.2f};\n")
                continue
            if in_initial_condition and in_V_in_initial_condition and '}' in stripped:
                in_U_in_initial_condition = False
                in_initial_condition = False
                f_out.write(line)
                continue

            # 3. CANOPY FILENAME
            if 'list {' in stripped:
                in_canopy_list = True
            if in_canopy_list and '}' in stripped:
                in_canopy_list = False
            if in_canopy_list and 'filename =' in line:
                forest_file = f"exp_{exp_id}_forest.txt"
                f_out.write(f'\t\tfilename = "{forest_file}";\n')
                continue

            # 4. PASSIVE TRACERS
            if 'passive_tracers' in stripped:
                in_passive_tracers = True
                passive_tracers_brace_count = 0
                f_out.write(line)
                continue
            
            if in_passive_tracers:
                passive_tracers_brace_count += line.count('{')
                passive_tracers_brace_count -= line.count('}')
                
                if passive_tracers_brace_count == 0:
                    in_passive_tracers = False
                    skip_lines_until_brace = False
                    f_out.write(line)
                    continue

                if skip_lines_until_brace:
                    continue

                if 'num =' in line:
                    f_out.write(f"\tnum = 16;\n") # 32
                    continue

                # When we hit the start of tracer definitions, replace them all
                if '# --- each tracer field defines diffusivity & surface values' in stripped or 'tracer_' in stripped:
                    for i in range(1, 17): # 33
                        # road_data[i] returns a list of rects
                        rects_list = road_data.get(i-1, [])
                        f_out.write(generate_tracer_block(i, rects_list))
                    skip_lines_until_brace = True
                    continue

            if not skip_lines_until_brace:
                f_out.write(line)

In [ ]:
# def process_config_file(exp_id, u_val, v_val, t_surf, road_data, input_path, output_path):
#     """
#     Reads the base config and writes the modified version.
#     Uses a state-machine logic to find and replace blocks.
#     """
#     with open(input_path, 'r') as f_in, open(output_path, 'w') as f_out:
        
#         # State trackers
#         in_geo_wind = False
#         in_init_theta = False
#         in_canopy_list = False
#         in_passive_tracers = False
        
#         # To skip existing tracer definitions in the base file
#         skip_lines_until_brace = False
#         passive_tracers_brace_count = 0

#         iterator = iter(f_in)
        
#         for line in iterator:
#             stripped = line.strip()
#             # --- 1. GEO WIND ---
#             if 'geo_wind' in stripped:
#                 in_geo_wind = True
#                 f_out.write(line)
#                 continue
            
#             if in_geo_wind:
#                 if '}' in stripped:
#                     in_geo_wind = False
#                 elif 'U =' in line and 'V =' in line:
#                     # Replace wind line
#                     f_out.write(f"\tU = {u_val:.2f}; V = {v_val:.2f};\n")
#                     continue

#             # --- 2. INITIAL CONDITIONS (THETA) ---
#             if 'initial_conditions' in stripped:
#                 # We need to look deeper into Theta
#                 pass 
            
#             if 'Theta {' in stripped: 
#                  # This check is simple; assumes Theta block is distinct
#                  # To be safer, we check context, but config structure is usually consistent
#                  # We assume this is inside initial_conditions based on typical file structure
#                  pass

#             # Detect specific line for surface_value inside Theta block
#             # (Simplified check: looks for the key variable)
#             if 'surface_value =' in line and 'initial boundary layer temperature' in line:
#                 f_out.write(f"\t\tsurface_value = {t_surf:.2f};	# initial boundary layer temperature [K]\n")
#                 continue

#             # --- 3. CANOPY FILENAME ---
#             if 'canopy' in stripped and '{' in stripped:
#                 pass # Just tracking, no specific state needed if we match filename line

#             if 'list {' in stripped:
#                 in_canopy_list = True
            
#             if in_canopy_list and '}' in stripped:
#                 in_canopy_list = False
                
#             if in_canopy_list and 'filename =' in line:
#                 # Replace forest filename
#                 forest_file = f"exp_{exp_id}_forest.txt"
#                 f_out.write(f'\t\tfilename = "{forest_file}";\n')
#                 continue

#             # --- 4. PASSIVE TRACERS ---
#             if 'passive_tracers' in stripped:
#                 in_passive_tracers = True
#                 passive_tracers_brace_count = 0
#                 f_out.write(line)
#                 continue
            
#             if in_passive_tracers:
#                 # Update brace count to detect end of block
#                 passive_tracers_brace_count += line.count('{')
#                 passive_tracers_brace_count -= line.count('}')
                
#                 if passive_tracers_brace_count == 0:
#                     in_passive_tracers = False
#                     skip_lines_until_brace = False
#                     f_out.write(line) # Write closing brace
#                     continue

#                 if skip_lines_until_brace:
#                     continue

#                 # Modify num
#                 if 'num =' in line:
#                     f_out.write(f"\tnum = 32;\n")
#                     continue

#                 # Identify the "cut point"
#                 # We want to keep global settings (emission.coeff, roads_out, factors, bc.parent_mode)
#                 # But remove existing 'tracer_1' and replace with our generated ones.
#                 # In config_test.txt, 'tracer_1' usually starts after 'bc.parent_mode'.
                
#                 # Check if we are starting definitions we want to overwrite
#                 if '# --- each tracer field defines diffusivity & surface values' in stripped or 'tracer_' in stripped:

#                     # STOP writing lines from source.
#                     # GENERATE our tracers.
#                     # Set flag to skip source lines until the end of passive_tracers block.
                    
#                     # 1. Generate new tracers
#                     for i in range(1, 33): # 1 to 32
#                         # Get coords from road data (0-indexed)
#                         rect = road_data.get(i-1, {'xmin':0,'xmax':1,'ymin':0,'ymax':1})
#                         f_out.write(generate_tracer_block(i, rect))
                    
#                     # 2. Start skipping
#                     skip_lines_until_brace = True
#                     continue

#             # Default: write line as is
#             if not skip_lines_until_brace:
#                 f_out.write(line)


In [6]:
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

road_files = glob.glob(os.path.join(FOREST_OUTPUT_DIR, "exp_*_roads.txt"))
print(f"Found {len(road_files)} experiment source files.")

data_records = []
for road_file in road_files:
    basename = os.path.basename(road_file)
    match = re.search(r'exp_(\d+)_roads.txt', basename)
    if not match:
        continue
    
    exp_id = int(match.group(1))
    
    # 1. Parse Roads (Gets lists of segments)
    road_data = parse_road_configs(road_file)
    if not road_data:
        print(f"Skipping exp {exp_id}: No road data found.")
        continue

    # 2. Generate Random Physics
    alpha = random.uniform(*ALPHA)
    dPdx = dPdX_CONST * np.cos(alpha)
    dPdy = dPdX_CONST * np.sin(alpha)

    u_val = VELOCITY_CONST* np.cos(alpha)
    v_val = VELOCITY_CONST* np.sin(alpha)
    #t_surf = random.uniform(*T_SURF_RANGE)

    # 3. Create Config
    out_filename = os.path.join(OUTPUT_DIR, f"config_exp_{exp_id}.txt")
    print(f"Generating Config {exp_id}: dPdx={dPdx:.6f}, dPdy={dPdy:.6f}, U={u_val:.2f}, V={v_val:.2f}")
    process_config_file(
        exp_id, dPdx, dPdy, u_val, v_val, road_data, BASE_CONFIG_PATH, out_filename
    )

    # 4. Collect Data for DataFrame
    # Теперь проходим по каждому трассеру и по КАЖДОМУ сегменту дороги внутри него
    for tracer_idx in range(16): # 32
        # Получаем список сегментов для данного трассера (0-31)
        rects_list = road_data.get(tracer_idx, [])
        
        # Если дорог нет, можно пропустить или добавить запись с нулями
        if not rects_list:
            continue

        # Добавляем строку для каждой дороги
        for segment_idx, rect in enumerate(rects_list, 1):
            record = {
                'experiment_id': exp_id,
                'tracer_id': tracer_idx + 1,  # В конфиге 1-16, 1-32
                'segment_id': segment_idx,    # Номер дороги внутри трассера (для различения)
                'dPdx': dPdx,
                'dPdy': dPdy,
                'u': u_val,
                'v': v_val,
                'x_min': rect['xmin'],
                'x_max': rect['xmax'],
                'y_min': rect['ymin'],
                'y_max': rect['ymax']
            }
            data_records.append(record)

# 5. Create and Save DataFrame
if data_records:
    df = pd.DataFrame(data_records)
    cols = ['experiment_id', 'tracer_id', 'segment_id','dPdx', 'dPdy', 'u', 'v', 'x_min', 'x_max', 'y_min', 'y_max']
    df = df[cols]
    
    csv_path = "experiments_metadata500_abs_2026_03_09.csv"
    df.to_csv(csv_path, index=False)
    print(f"\nMetadata saved to {csv_path}")
    print(f"Total rows: {len(df)}")
    print(df.head())
else:
    print("No data collected.")

Found 500 experiment source files.
Generating Config 49: dPdx=0.000250, dPdy=-0.000009, U=-3.00, V=0.10
Generating Config 423: dPdx=0.000234, dPdy=0.000087, U=-2.81, V=-1.05
Generating Config 313: dPdx=-0.000092, dPdy=0.000233, U=1.10, V=-2.79
Generating Config 17: dPdx=0.000107, dPdy=-0.000226, U=-1.29, V=2.71
Generating Config 160: dPdx=0.000108, dPdy=-0.000226, U=-1.29, V=2.71
Generating Config 388: dPdx=0.000087, dPdy=0.000234, U=-1.05, V=-2.81
Generating Config 273: dPdx=0.000249, dPdy=-0.000022, U=-2.99, V=0.26
Generating Config 490: dPdx=0.000159, dPdy=-0.000193, U=-1.90, V=2.32
Generating Config 205: dPdx=0.000207, dPdy=-0.000140, U=-2.49, V=1.68
Generating Config 148: dPdx=0.000120, dPdy=-0.000219, U=-1.44, V=2.63
Generating Config 455: dPdx=-0.000124, dPdy=-0.000217, U=1.48, V=2.61
Generating Config 365: dPdx=0.000169, dPdy=0.000184, U=-2.03, V=-2.21
Generating Config 61: dPdx=-0.000196, dPdy=0.000155, U=2.36, V=-1.86
Generating Config 116: dPdx=0.000094, dPdy=0.000232, U=-1.